In [ ]:
import pandas as pd
import unicodedata

anvisa = pd.read_csv("tabela_anvisa_limpa.csv", sep=";", encoding="utf-8-sig")
drogasil = pd.read_csv("medicamentos_base2_3.csv")


In [ ]:
def limpar_texto(t):
    if pd.isna(t):
        return ""
    t = str(t).lower().strip()
    # acentos
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    # facilitar o match
    t = t.replace("mg", " mg")
    t = t.replace("ml", " ml")
    t = t.replace("-", " ")
    t = " ".join(t.split())
    return t

In [ ]:
# removendo as linhas sem substancias, se nao quebra o join
drogasil_sem_subst = drogasil[drogasil["substancia"].isna()].copy()
drogasil = drogasil[~drogasil["substancia"].isna()].copy()

print("Drogasil sem substância:", len(drogasil_sem_subst))
print("Drogasil com substância:", len(drogasil))

In [ ]:
# limpeza e norm
#ANVISA 
anvisa["farmaco_limpo"] = anvisa["Fármaco"].apply(limpar_texto)

#DROGASIL
drogasil["substancia_limpa"] = drogasil["substancia"].apply(limpar_texto)

# lista de fármacos únicos da ANVISA pra usar no """like"""
farmacos = anvisa["farmaco_limpo"].dropna().unique().tolist()
print("Qtde fármacos únicos ANVISA:", len(farmacos))

In [ ]:
# join com substr
# a cada subs limpa da drogasil
# retorna a lista de farmacos da anvisa que batem
def mapear_farmacos(sub):
    if not isinstance(sub, str) or not sub.strip():
        return []
    return [f for f in farmacos if f in sub]


#aplicando essa praga
# criando a lista que falei acima e conta quantos darmacos cada med tem
drogasil["farmacos_encontrados"] = drogasil["substancia_limpa"].apply(mapear_farmacos)
drogasil["qtde_farmacos"] = drogasil["farmacos_encontrados"].apply(len)

print("Distribuição de qtde de fármacos por medicamento:")
print(drogasil["qtde_farmacos"].value_counts())

In [ ]:

#separando oq tem e oq nao tem farmanco
drogasil["tem_farmaco"] = drogasil["qtde_farmacos"] > 0

# oq nao foi mapeado é natural
drogasil_ok = drogasil[drogasil["tem_farmaco"]].copy()
drogasil_sem_farmaco = drogasil[~drogasil["tem_farmaco"]].copy()

print("Com fármaco mapeado:", len(drogasil_ok))
print("Sem fármaco mapeado:", len(drogasil_sem_farmaco))

In [ ]:
#expande os meds que tem mais de um farmaco
# se tem mais de um farmaco, mais de uma linha
drog_explo = drogasil_ok.explode("farmacos_encontrados")


# join da anvida com a drogasil
df_join = drog_explo.merge(
    anvisa,
    left_on="farmacos_encontrados",
    right_on="farmaco_limpo",
    how="left"
)
# linha da drogasil com linha da anvisa que corrsponde ao farmaco explodido
print("Shape do join many-to-many:", df_join.shape)

In [ ]:
colunas_excluir = [
    "Fármaco",
    "Subgrupo terapêutico ou farmacológico",
    "Forma farmacêutica",
    "Concentração máxima",
    "Indicação terapêutica simplificada",
    "indicacao_limpa",
    "indicacoes_list",
    "farmaco_limpo",
]


dummies_cols = [
    col for col in anvisa.columns
    if col not in colunas_excluir and anvisa[col].dtype != "O"
]

print("Qtd colunas de indicação/dummies:", len(dummies_cols))

In [ ]:
id_col = "idMedicamentos"

base_cols = [
    "nomeMedicamento",
    "preco",
    "marca",
    "quantidade",
    "dosagem",
    "substancia",
    "substancia_limpa",
]

agg_dict = {col: "first" for col in base_cols if col in df_join.columns}

# lista de fármacos distintos por medicamento
agg_dict["farmacos_encontrados"] = lambda x: sorted(
    set(f for f in x if isinstance(f, str))
)

# dummies: se qualquer fármaco tiver 1, o medicamento recebe 1
for col in dummies_cols:
    if col in df_join.columns:
        agg_dict[col] = "max"

df_final = df_join.groupby(id_col).agg(agg_dict).reset_index()

print("Shape df_final (1 linha por medicamento):", df_final.shape)
df_final.head()

In [ ]:
df_final.to_csv("dataset_final_medicamentos.csv", index=False, encoding="utf-8-sig")


In [ ]:
df_final.sample(10)[["nomeMedicamento", "substancia", "farmacos_encontrados"]]
